# Actividad 03 — Semana 03: SQL Avanzado

**Semana:** 03  
**Tema:** CTEs, subqueries anidadas y window functions en Spark SQL  
**Estudiante:** Daniel Guzmán  
**Notebook:** sql_avanzado_daniel  

## Objetivo

Aplicar SQL avanzado sobre tablas Delta usando:

- CTEs (`WITH`)
- CTEs encadenadas
- Window functions
- `ROW_NUMBER`, `RANK`, `DENSE_RANK`
- `LAG`, `LEAD`
- Acumulados con `SUM OVER`
- Media móvil
- `NTILE`
- Combinación de CTEs + window functions

La tabla principal usada será:

- `workspace.silver.transactions_daniel`

Se usa la tabla sufijada con `_daniel` para respetar la convención del entorno compartido.

In [0]:
USE CATALOG workspace;

SHOW TABLES IN silver;

In [0]:
DESCRIBE TABLE silver.transactions_daniel;

In [0]:
SELECT
    COUNT(*) AS total_transacciones,
    COUNT(DISTINCT user_id) AS total_usuarios,
    SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
    SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
    SUM(CASE WHEN is_fraud IS NULL THEN 1 ELSE 0 END) AS sin_label
FROM silver.transactions_daniel;

## Conteo base

La tabla `silver.transactions_daniel` contiene **13,305,915 transacciones** de **1,219 usuarios**.

Se identificaron:

- **13,332 transacciones fraudulentas**
- **8,914,963 transacciones etiquetadas**
- **4,390,952 transacciones sin label**

Por esta razón, cuando se calculen tasas de fraude se usará como denominador `transacciones_etiquetadas`, no el total completo de transacciones.

In [0]:
-- SIN CTE: usuarios con más fraude que el promedio de fraudes por usuario

SELECT user_id, total_fraudes
FROM (
    SELECT
        user_id,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes
    FROM silver.transactions_daniel
    GROUP BY user_id
) resumen
WHERE total_fraudes > (
    SELECT AVG(fraudes_por_usuario)
    FROM (
        SELECT
            user_id,
            SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraudes_por_usuario
        FROM silver.transactions_daniel
        GROUP BY user_id
    ) sub
)
ORDER BY total_fraudes DESC
LIMIT 20;

In [0]:
-- CON CTEs: mismo resultado, más legible

WITH fraudes_por_usuario AS (
    SELECT
        user_id,
        COUNT(*) AS total_transacciones,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
        SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
        ROUND(
            SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
            / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
            4
        ) AS tasa_fraude_pct
    FROM silver.transactions_daniel
    GROUP BY user_id
),
promedio_fraudes AS (
    SELECT AVG(total_fraudes) AS media_fraudes
    FROM fraudes_por_usuario
)
SELECT
    f.user_id,
    f.total_transacciones,
    f.total_fraudes,
    f.transacciones_etiquetadas,
    f.tasa_fraude_pct,
    p.media_fraudes,
    ROUND(f.total_fraudes - p.media_fraudes, 2) AS desviacion_vs_media
FROM fraudes_por_usuario f
CROSS JOIN promedio_fraudes p
WHERE f.total_fraudes > p.media_fraudes
ORDER BY f.total_fraudes DESC
LIMIT 20;

## Parte 1.1 — CTE básica vs subquery

La consulta con subqueries funciona, pero es más difícil de leer porque repite varias veces la lógica de fraude por usuario.

Con CTEs, la consulta se vuelve más clara porque cada paso tiene un nombre:

- `fraudes_por_usuario`: calcula las métricas por usuario.
- `promedio_fraudes`: calcula la media global.
- Consulta final: compara cada usuario contra la media.

Se pueden encadenar varias CTEs dentro de un mismo `WITH`. Una CTE puede referenciar a otra CTE definida antes, como ocurre cuando `promedio_fraudes` lee desde `fraudes_por_usuario`.

In [0]:
WITH total_global AS (
    SELECT
        COUNT(*) AS tx_totales,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraudes_totales,
        SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS tx_etiquetadas,
        ROUND(
            SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
            / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
            4
        ) AS tasa_global
    FROM silver.transactions_daniel
),
por_categoria AS (
    SELECT
        merchant_category,
        COUNT(*) AS tx_categoria,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraudes_categoria,
        SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS tx_etiquetadas_categoria,
        ROUND(
            SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
            / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
            4
        ) AS tasa_categoria
    FROM silver.transactions_daniel
    WHERE merchant_category IS NOT NULL
    GROUP BY merchant_category
)
SELECT
    c.merchant_category,
    c.tx_categoria,
    c.fraudes_categoria,
    c.tx_etiquetadas_categoria,
    c.tasa_categoria,
    g.tasa_global,
    ROUND(c.tasa_categoria - g.tasa_global, 4) AS diferencia_vs_global,
    CASE
        WHEN c.tasa_categoria > g.tasa_global * 1.5 THEN 'Alto riesgo'
        WHEN c.tasa_categoria > g.tasa_global THEN 'Riesgo elevado'
        ELSE 'Riesgo normal'
    END AS clasificacion_riesgo
FROM por_categoria c
CROSS JOIN total_global g
ORDER BY c.tasa_categoria DESC;

## Parte 1.2 — CTEs múltiples encadenadas

Se usaron dos CTEs:

- `total_global`: calcula la tasa global de fraude.
- `por_categoria`: calcula la tasa de fraude por categoría de comercio.

Luego se usa un `CROSS JOIN` para comparar cada categoría contra la tasa global.

Este patrón es útil porque permite separar el cálculo global del cálculo por dimensión, dejando la consulta más legible y fácil de mantener.

Las tasas se calcularon usando únicamente transacciones etiquetadas para evitar que los registros con `is_fraud IS NULL` diluyan el resultado.

In [0]:
WITH gasto_usuario AS (
    SELECT
        user_id,
        card_type,
        ROUND(SUM(amount_abs), 2) AS gasto_total,
        COUNT(*) AS num_transacciones
    FROM silver.transactions_daniel
    WHERE card_type IS NOT NULL
    GROUP BY user_id, card_type
)
SELECT
    user_id,
    card_type,
    gasto_total,
    num_transacciones,
    ROW_NUMBER() OVER (PARTITION BY card_type ORDER BY gasto_total DESC) AS row_num,
    RANK() OVER (PARTITION BY card_type ORDER BY gasto_total DESC) AS rank_gasto,
    DENSE_RANK() OVER (PARTITION BY card_type ORDER BY gasto_total DESC) AS dense_rank_gasto
FROM gasto_usuario
QUALIFY ROW_NUMBER() OVER (PARTITION BY card_type ORDER BY gasto_total DESC) <= 5
ORDER BY card_type, row_num;

## Parte 2.1 — ROW_NUMBER, RANK y DENSE_RANK

Se usaron window functions para rankear usuarios por gasto total dentro de cada tipo de tarjeta.

`PARTITION BY card_type` separa el ranking por tipo de tarjeta.  
`ORDER BY gasto_total DESC` ordena los usuarios de mayor a menor gasto dentro de cada partición.

### Diferencia entre ROW_NUMBER, RANK y DENSE_RANK

- `ROW_NUMBER`: asigna un número único a cada fila, incluso si hay empates.
- `RANK`: asigna el mismo ranking a valores empatados, pero deja saltos en la numeración.
- `DENSE_RANK`: asigna el mismo ranking a valores empatados, pero no deja saltos.

### QUALIFY

`QUALIFY` permite filtrar resultados después de calcular una window function.  
Es útil porque evita envolver la consulta en otra subquery solo para filtrar por `ROW_NUMBER`, `RANK` o `DENSE_RANK`.

No es SQL estándar universal; es una extensión soportada por motores como Databricks/Spark SQL y otros sistemas analíticos modernos.

In [0]:
WITH tx_ordenadas AS (
    SELECT
        user_id,
        transaction_date,
        amount,
        merchant_id,
        merchant_category,
        is_fraud
    FROM silver.transactions_daniel
    WHERE user_id IS NOT NULL
)
SELECT
    user_id,
    transaction_date,
    amount,
    merchant_id,
    merchant_category,
    LAG(amount, 1) OVER (PARTITION BY user_id ORDER BY transaction_date) AS monto_anterior,
    LEAD(amount, 1) OVER (PARTITION BY user_id ORDER BY transaction_date) AS monto_siguiente,
    ROUND(
        amount - LAG(amount, 1) OVER (PARTITION BY user_id ORDER BY transaction_date),
        2
    ) AS variacion_vs_anterior,
    is_fraud
FROM tx_ordenadas
ORDER BY user_id, transaction_date
LIMIT 50;

In [0]:
WITH tx_con_lag AS (
    SELECT
        user_id,
        transaction_date,
        amount,
        merchant_id,
        merchant_category,
        is_fraud,
        LAG(amount, 1) OVER (PARTITION BY user_id ORDER BY transaction_date) AS monto_anterior
    FROM silver.transactions_daniel
)
SELECT
    user_id,
    transaction_date,
    amount,
    monto_anterior,
    ROUND(amount - monto_anterior, 2) AS variacion_vs_anterior,
    merchant_id,
    merchant_category,
    is_fraud
FROM tx_con_lag
WHERE monto_anterior IS NOT NULL
  AND amount > monto_anterior * 2
  AND amount > 100
ORDER BY amount DESC
LIMIT 20;

In [0]:
WITH usuario_muestra AS (
    SELECT user_id
    FROM silver.transactions_daniel
    WHERE user_id IS NOT NULL
    LIMIT 1
)
SELECT
    t.user_id,
    t.transaction_date,
    t.amount,
    SUM(t.amount) OVER (
        PARTITION BY t.user_id
        ORDER BY t.transaction_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS gasto_acumulado,
    AVG(t.amount) OVER (
        PARTITION BY t.user_id
        ORDER BY t.transaction_date
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS media_movil_3
FROM silver.transactions_daniel t
INNER JOIN usuario_muestra u
    ON t.user_id = u.user_id
ORDER BY t.transaction_date;

In [0]:
WITH gasto_por_usuario AS (
    SELECT
        user_id,
        ROUND(SUM(amount_abs), 2) AS gasto_total,
        COUNT(*) AS num_transacciones,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes
    FROM silver.transactions_daniel
    WHERE user_id IS NOT NULL
    GROUP BY user_id
),
usuarios_cuartiles AS (
    SELECT
        user_id,
        gasto_total,
        num_transacciones,
        total_fraudes,
        NTILE(4) OVER (ORDER BY gasto_total) AS cuartil_gasto,
        NTILE(10) OVER (ORDER BY gasto_total) AS decil_gasto
    FROM gasto_por_usuario
)
SELECT
    user_id,
    gasto_total,
    num_transacciones,
    total_fraudes,
    cuartil_gasto,
    decil_gasto
FROM usuarios_cuartiles
ORDER BY gasto_total DESC
LIMIT 30;

In [0]:
WITH gasto_por_usuario AS (
    SELECT
        user_id,
        ROUND(SUM(amount_abs), 2) AS gasto_total,
        COUNT(*) AS num_transacciones,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes
    FROM silver.transactions_daniel
    WHERE user_id IS NOT NULL
    GROUP BY user_id
),
usuarios_cuartiles AS (
    SELECT
        user_id,
        gasto_total,
        num_transacciones,
        total_fraudes,
        NTILE(4) OVER (ORDER BY gasto_total) AS cuartil_gasto
    FROM gasto_por_usuario
),
fraude_total AS (
    SELECT SUM(total_fraudes) AS fraudes_globales
    FROM usuarios_cuartiles
)
SELECT
    cuartil_gasto,
    COUNT(*) AS total_usuarios,
    SUM(total_fraudes) AS fraudes_cuartil,
    ROUND(SUM(total_fraudes) / MAX(fraudes_globales) * 100, 4) AS pct_fraude_total
FROM usuarios_cuartiles
CROSS JOIN fraude_total
GROUP BY cuartil_gasto
ORDER BY cuartil_gasto;

## Parte 2.3 y 2.4 — Acumulados, media móvil y NTILE

Se usaron window functions con `ROWS BETWEEN` para calcular métricas progresivas por usuario.

- `SUM(amount) OVER (...)` calcula el gasto acumulado.
- `AVG(amount) OVER (... ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` calcula una media móvil de 3 transacciones.
- `NTILE(4)` divide los usuarios en cuartiles según su gasto total.
- `NTILE(10)` divide los usuarios en deciles.

El cuartil de mayor gasto permite analizar si el fraude se concentra en usuarios de mayor volumen transaccional.

En el análisis por cuartiles, el cuartil de mayor gasto (`cuartil_gasto = 4`) concentra **3,373 fraudes**, equivalentes al **25.3% del fraude total**.

In [0]:
WITH volumen_comercio_mes AS (
    SELECT
        merchant_id,
        merchant_category,
        mes,
        anio,
        COUNT(*) AS num_transacciones,
        ROUND(SUM(amount_abs), 2) AS monto_total
    FROM silver.transactions_daniel
    WHERE merchant_category IS NOT NULL
    GROUP BY merchant_id, merchant_category, mes, anio
),
ranking_en_categoria AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY merchant_category, anio, mes
            ORDER BY monto_total DESC
        ) AS rank_mes
    FROM volumen_comercio_mes
)
SELECT
    merchant_category,
    anio,
    mes,
    merchant_id,
    num_transacciones,
    monto_total
FROM ranking_en_categoria
WHERE rank_mes = 1
ORDER BY anio, mes, merchant_category;

## Parte 3 — CTE + Window Function combinados

Se combinó una CTE con una window function para identificar el comercio con mayor volumen por categoría y mes.

Primero, la CTE `volumen_comercio_mes` calcula el volumen mensual por `merchant_id`, categoría, año y mes.

Luego, la CTE `ranking_en_categoria` aplica `ROW_NUMBER()` particionando por `merchant_category`, `anio` y `mes`, y ordenando por `monto_total` descendente.

Finalmente, se filtra `rank_mes = 1` para obtener el comercio principal de cada categoría en cada mes.

Como la tabla Silver no contiene `merchant_name`, se usó `merchant_id` como identificador del comercio.

In [0]:
WITH tx_numeradas AS (
    SELECT
        user_id,
        transaction_id,
        transaction_date,
        is_fraud,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY transaction_date
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY user_id
        ) AS total_tx_usuario
    FROM silver.transactions_daniel
    WHERE user_id IS NOT NULL
),
tx_mitad AS (
    SELECT
        user_id,
        transaction_id,
        transaction_date,
        is_fraud,
        CASE
            WHEN rn <= total_tx_usuario / 2 THEN 'primera_mitad'
            ELSE 'segunda_mitad'
        END AS mitad
    FROM tx_numeradas
),
resumen_mitad AS (
    SELECT
        user_id,
        mitad,
        COUNT(*) AS total_transacciones,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS total_fraudes,
        SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) AS transacciones_etiquetadas,
        ROUND(
            SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END)
            / SUM(CASE WHEN is_fraud IN (0, 1) THEN 1 ELSE 0 END) * 100,
            4
        ) AS tasa_fraude_pct
    FROM tx_mitad
    GROUP BY user_id, mitad
),
comparacion_usuario AS (
    SELECT
        user_id,
        MAX(CASE WHEN mitad = 'primera_mitad' THEN tasa_fraude_pct END) AS tasa_primera_mitad,
        MAX(CASE WHEN mitad = 'segunda_mitad' THEN tasa_fraude_pct END) AS tasa_segunda_mitad,
        MAX(CASE WHEN mitad = 'primera_mitad' THEN total_fraudes END) AS fraudes_primera_mitad,
        MAX(CASE WHEN mitad = 'segunda_mitad' THEN total_fraudes END) AS fraudes_segunda_mitad,
        MAX(CASE WHEN mitad = 'primera_mitad' THEN transacciones_etiquetadas END) AS etiquetadas_primera_mitad,
        MAX(CASE WHEN mitad = 'segunda_mitad' THEN transacciones_etiquetadas END) AS etiquetadas_segunda_mitad
    FROM resumen_mitad
    GROUP BY user_id
)
SELECT
    user_id,
    tasa_primera_mitad,
    tasa_segunda_mitad,
    ROUND(tasa_segunda_mitad - tasa_primera_mitad, 4) AS aumento_tasa_pct,
    fraudes_primera_mitad,
    fraudes_segunda_mitad,
    etiquetadas_primera_mitad,
    etiquetadas_segunda_mitad
FROM comparacion_usuario
WHERE tasa_segunda_mitad > tasa_primera_mitad
  AND etiquetadas_primera_mitad >= 100
  AND etiquetadas_segunda_mitad >= 100
ORDER BY aumento_tasa_pct DESC
LIMIT 20;

## Pregunta 1 — Usuarios que aceleran en fraude

Se usó `ROW_NUMBER()` y `COUNT(*) OVER` para ordenar cronológicamente las transacciones de cada usuario y dividir su historial en dos mitades.

Luego se comparó la tasa de fraude de la primera mitad contra la segunda mitad.

Este análisis permite identificar usuarios cuyo comportamiento se vuelve más riesgoso con el tiempo. Se filtraron usuarios con al menos 100 transacciones etiquetadas en cada mitad para evitar conclusiones basadas en muy pocos datos.

In [0]:
WITH fraude_categoria_hora AS (
    SELECT
        merchant_category,
        hora,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraudes_hora
    FROM silver.transactions_daniel
    WHERE merchant_category IS NOT NULL
      AND hora IS NOT NULL
    GROUP BY merchant_category, hora
),
fraude_con_total AS (
    SELECT
        merchant_category,
        hora,
        fraudes_hora,
        SUM(fraudes_hora) OVER (
            PARTITION BY merchant_category
        ) AS fraudes_categoria
    FROM fraude_categoria_hora
),
ranking_hora AS (
    SELECT
        merchant_category,
        hora,
        fraudes_hora,
        fraudes_categoria,
        ROUND(fraudes_hora / fraudes_categoria * 100, 4) AS pct_fraude_en_hora,
        ROW_NUMBER() OVER (
            PARTITION BY merchant_category
            ORDER BY fraudes_hora DESC
        ) AS rank_hora
    FROM fraude_con_total
    WHERE fraudes_categoria > 0
)
SELECT
    merchant_category,
    hora AS hora_principal,
    fraudes_hora,
    fraudes_categoria,
    pct_fraude_en_hora
FROM ranking_hora
WHERE rank_hora = 1
  AND pct_fraude_en_hora > 60
ORDER BY pct_fraude_en_hora DESC, fraudes_categoria DESC;

## Pregunta 2 — Categorías con fraude concentrado en una sola hora

Se agrupó el fraude por `merchant_category` y `hora`.

Después se usó `SUM(...) OVER (PARTITION BY merchant_category)` para calcular el fraude total de cada categoría sin perder el detalle por hora.

Finalmente, se aplicó `ROW_NUMBER()` para encontrar la hora con más fraude dentro de cada categoría y filtrar aquellas donde más del 60% del fraude ocurre en una sola hora.

Este análisis ayuda a detectar patrones horarios muy concentrados que podrían ser señales de comportamiento anómalo.

In [0]:
WITH tx_ordenadas AS (
    SELECT
        user_id,
        transaction_id,
        transaction_date,
        amount_abs,
        merchant_category,
        is_fraud,
        LAG(transaction_date, 1) OVER (
            PARTITION BY user_id
            ORDER BY transaction_date
        ) AS transaccion_anterior,
        LAG(is_fraud, 1) OVER (
            PARTITION BY user_id
            ORDER BY transaction_date
        ) AS fraude_anterior
    FROM silver.transactions_daniel
    WHERE user_id IS NOT NULL
),
fraudes_con_vecino AS (
    SELECT
        user_id,
        transaction_id,
        transaction_date,
        transaccion_anterior,
        ROUND(
            (UNIX_TIMESTAMP(transaction_date) - UNIX_TIMESTAMP(transaccion_anterior)) / 60,
            2
        ) AS minutos_desde_anterior,
        amount_abs,
        merchant_category,
        is_fraud,
        fraude_anterior
    FROM tx_ordenadas
    WHERE is_fraud = 1
      AND transaccion_anterior IS NOT NULL
),
fraudes_clasificados AS (
    SELECT
        *,
        CASE
            WHEN minutos_desde_anterior < 5 THEN 'menos_5_min'
            WHEN minutos_desde_anterior < 30 THEN '5_30_min'
            WHEN minutos_desde_anterior < 60 THEN '30_60_min'
            WHEN minutos_desde_anterior < 1440 THEN '1_24_horas'
            ELSE 'mas_24_horas'
        END AS rango_tiempo_vecino
    FROM fraudes_con_vecino
)
SELECT
    rango_tiempo_vecino,
    COUNT(*) AS total_fraudes,
    ROUND(AVG(minutos_desde_anterior), 2) AS promedio_minutos_desde_anterior,
    ROUND(MIN(minutos_desde_anterior), 2) AS min_minutos,
    ROUND(MAX(minutos_desde_anterior), 2) AS max_minutos
FROM fraudes_clasificados
GROUP BY rango_tiempo_vecino
ORDER BY
    CASE rango_tiempo_vecino
        WHEN 'menos_5_min' THEN 1
        WHEN '5_30_min' THEN 2
        WHEN '30_60_min' THEN 3
        WHEN '1_24_horas' THEN 4
        ELSE 5
    END;

## Pregunta 3 — Efecto vecino

Se usó `LAG(transaction_date)` para obtener la transacción anterior de cada usuario.

Después se calcularon los minutos transcurridos entre cada transacción fraudulenta y la transacción inmediatamente anterior del mismo usuario.

El objetivo es observar si las transacciones fraudulentas tienden a ocurrir cerca de otras transacciones o si aparecen aisladas en el tiempo.

Este análisis puede ayudar a detectar patrones de ráfaga, donde varias operaciones ocurren en intervalos cortos.

## Conclusiones del análisis final

### Pregunta 1 — Usuarios que aceleran en fraude

Se identificaron usuarios cuya tasa de fraude aumentó en la segunda mitad de su historial transaccional.

Un ejemplo visible es el usuario `1725`, que pasó de **0%** en la primera mitad a **1.0098%** en la segunda mitad, acumulando **38 fraudes** en esa segunda etapa.

Esto muestra que algunos usuarios pueden cambiar su comportamiento con el tiempo, por lo que no basta con analizar solo el histórico completo; también es útil comparar periodos internos del comportamiento del usuario.

### Pregunta 2 — Categorías con fraude concentrado en una sola hora

Se encontraron categorías donde el 100% del fraude ocurrió en una única hora:

- `Podiatrists`, con fraude concentrado en la hora **16**.
- `Recreational Sports, Clubs`, con fraude concentrado en la hora **19**.

Sin embargo, cada una tiene solo **1 fraude total**, por lo que el resultado debe interpretarse con cuidado. Aunque cumple la regla de concentración mayor al 60%, el bajo volumen no permite concluir que sea un patrón fuerte.

### Pregunta 3 — Efecto vecino

El análisis con `LAG(transaction_date)` mostró que la mayoría de las transacciones fraudulentas ocurren dentro del rango de **1 a 24 horas** desde la transacción anterior del mismo usuario.

Distribución observada:

- `menos_5_min`: 967 fraudes
- `5_30_min`: 3,288 fraudes
- `30_60_min`: 1,626 fraudes
- `1_24_horas`: 7,131 fraudes
- `mas_24_horas`: 320 fraudes

Esto sugiere que muchas transacciones fraudulentas no ocurren completamente aisladas, sino relativamente cerca de otra transacción previa del mismo usuario.